# Calibración estratificada y cobertura

**Nivel 2 — Análisis aplicado**  
**Prerrequisitos:** nivel 1 y arreglos/logaritmos; 35 min.

## Objetivos

Siga la evidencia desde el contrato fuente hasta el resultado. Declare unidades y supuestos, reproduzca el cálculo e identifique qué inferencia no respalda la evidencia.

## Preparación

Usa módulos y registros versionados desde la raíz descubierta; no descarga ni modifica fuentes. Instale versiones exactas: `pip install numpy==2.2.6 matplotlib==3.11.2` `scipy==1.15.2` es requerido por el operador directo MT.

## Datos

Registro medido solo QC y artefacto sintético de calibración separado; no mezcle estas clases de evidencia.

## Método

Este artefacto es sintético: tierra 1D de dos capas, `[180,20] Ω·m`, espesor superior 450 m, 24 frecuencias, 48 realizaciones ruidosas independientes y 128 réplicas bootstrap por realización. La cobertura empírica es `C=N⁻¹Σ1[ρreal∈IC]`; la unidad independiente es la realización, no cada réplica. Compare cobertura con 95 % nominal y límites Wilson guardados, e inspeccione anchuras en Ω·m. El intervalo depende de espesor fijo, errores independientes gaussianos real/imaginario, cotas y regularización. No garantiza cobertura en campo; alta cobertura puede coexistir con sesgo o capas mal resueltas.

Los parámetros son log-resistividades: positivos por construcción y pasos iguales son cambios multiplicativos iguales. Residuos complejos usan sigma por componente real; ambas partes aportan al ajuste. Frecuencias no son muestras independientes de profundidad: kernels se solapan y un período no identifica una interfaz. Sensibilidad local y cobertura dependen de parametrización fija y no descartan equivalencia global.

## Resultados

Lea ejes físicos y unidades; distinga ajuste, error contra verdad sintética y acuerdo con etiquetas humanas. El recálculo debe concordar con el recibo. Un modelo graficado no demuestra unicidad, transferencia a campo ni resolución.

### Diagnostic 1

Inspeccione el recibo y procedencia del caso.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib', 'scipy']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Inspect synthetic calibration provenance and independent realization rows.
import json
from pathlib import Path
cal=json.loads((root/"data/derived/v2/edi/calibration-two-layer.json").read_text(encoding="utf-8"))
print(cal["kind"],cal["calibration_model"],"Ω·m; thickness",cal["fixed_thickness_m"],"m; frequencies",len(cal["frequencies_hz"]))
print("realizations",cal["realizations"],"bootstrap each",cal["bootstrap_samples_per_realization"],"nominal",cal["confidence"],"coverage",cal["coverage_per_layer"])

### Diagnostic 2

Calcule la comparación y grafíquela con unidades físicas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Recompute coverage and median interval widths from the 48 recorded fits.
import matplotlib.pyplot as plt
rows=cal["rows"]; truth=np.asarray(cal["calibration_model"],float); cover=np.asarray([r["covered"] for r in rows],bool); ci=np.asarray([r["interval_ohm_m"] for r in rows],float)
cov=cover.mean(0); width=np.median(ci[:,1,:]-ci[:,0,:],axis=0); print("coverage",cov,"; median interval width Ω·m",width)
fig,ax=plt.subplots(1,2,figsize=(8,3.5),constrained_layout=True); x=np.arange(2); ax[0].bar(x,cov); ax[0].axhline(cal["confidence"],ls="--",color="crimson"); ax[0].errorbar(x,cov,yerr=[np.maximum(0,cov-np.asarray(cal["coverage_wilson95_lower"])),np.maximum(0,np.asarray(cal["coverage_wilson95_upper"])-cov)],fmt="none",capsize=4,color="black"); ax[0].set(xticks=x,xticklabels=["upper","lower"],ylim=(0,1.05),ylabel="Coverage fraction",title="Realization-level coverage")
ax[1].bar(x,width,color="#e1812c"); ax[1].set(xticks=x,xticklabels=["upper","lower"],ylabel="Median interval width (Ω·m)",title="Conditional uncertainty"); [a.grid(axis="y",alpha=.2) for a in ax]; plt.show()

## Resumen

La conclusión depende de geometría, adquisición, ruido, partición, parametrización y derechos de fuente. Conserve negativos y nombre la observación o control siguiente.

## Referencias

- [https://github.com/MTgeophysics/mtpy-v2](https://github.com/MTgeophysics/mtpy-v2)
- [https://www.usgs.gov/data/magnetotelluric-data-clear-lake-region-northern-california](https://www.usgs.gov/data/magnetotelluric-data-clear-lake-region-northern-california)